# Phi-3 Mini Personal Finance — End-to-End Fine-Tuning

Fine-tune Microsoft's **Phi-3 Mini (3.8B)** on personal finance Q&A, evaluate it, fix failures, merge, deploy a Gradio UI, and optionally deploy to **GCP Vertex AI**.

**Stack:** Unsloth · QLoRA · TRL · BitsAndBytes · Gradio · GCP Vertex AI

| Section | What happens | Time |
|---|---|---|
| **1 – Setup** | GPU check, install libraries | 5 min |
| **2 – Model** | Load Phi-3 Mini + LoRA adapters | 2 min |
| **3 – Dataset** | Load CSV, format into Phi-3 template | 1 min |
| **4 – Train v1** | First fine-tuning run | 30–60 min |
| **5 – Inference** | Single + multi-turn chat | instant |
| **6 – Evaluate** | 30-prompt evaluation, manual scoring | 15 min |
| **7 – Retrain v2** | Fix failures, retrain on improved data | 30–45 min |
| **8 – Merge** | Merge LoRA into base model | 5 min |
| **9 – Gradio UI** | Launch shareable chat interface | 1 min |
| **10 – GCP** | Deploy to Vertex AI (CPU or GPU) | 15–20 min |

**Every cell calls one function from `main.py`. Run top to bottom.**

In [1]:
# Run BEFORE import main
!pip install --quiet \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "bitsandbytes>=0.43.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "accelerate>=0.30.0" \
    "datasets>=2.18.0" \
    "transformers>=4.40.0" \
    "gradio>=4.26.0"

!pip install --upgrade --no-deps --force-reinstall --no-cache-dir unsloth unsloth_zoo

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.4 MB/s eta 

---
## Session Start — Run This First Every Time
Uploads `main.py` and authenticates with GCP.

In [2]:
# SESSION — Upload main.py (lost on every Colab restart)
from google.colab import files
print('Upload main.py from your computer ...')
files.upload()

import main
print('✅ main.py loaded')

Upload main.py from your computer ...


Saving main.py to main.py
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ main.py loaded


---
## Section 1 — Environment Setup
In Colab: **Runtime → Change runtime type → GPU (A100 or T4)**

In [3]:
# 1a — Check GPU
gpu_info = main.check_gpu()
print(gpu_info)

Sun Aug  2 10:41:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P0             26W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# 1b — Install libraries (~3 min, first run only)
main.install_libraries()

📦 Installing libraries …
✅ All libraries installed!


---
## Section 2 — Load Model
Loads Phi-3 Mini (3.8B) in 4-bit quantization (~6 GB VRAM).

In [5]:
# 2a — Load Phi-3 Mini in 4-bit
model, tokenizer = main.load_base_model()

🧠 Loading unsloth/Phi-3-mini-4k-instruct …
==((====))==  Unsloth 2026.7.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded  (dtype=torch.float16)


In [6]:
# 2b — Apply LoRA adapters (~1-5% of params become trainable)
model = main.apply_lora(model)

Unsloth 2026.7.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA applied — 29,884,416 / 2,039,024,640 params trainable (1.47%)


---
## Section 3 — Dataset

> Upload `personal_finance_qa.csv` to Colab before running.
> Required columns: **Question**, **Answer** (~400 rows)

In [7]:
# 3a — Load CSV
df = main.load_csv()   # reads personal_finance_qa.csv
df.head(3)

📂 Loaded 400 rows  |  Columns: ['Question', 'Answer']
✅ 400 clean rows ready.


,Question,Answer
0,What is a budget?,A plan that allocates your income toward expen...
1,What is the 50/30/20 rule?,A budgeting guideline: 50% of after-tax income...
2,What are 'needs' in the 50/30/20 rule?,"Essential expenses you cannot avoid, such as r..."


In [8]:
# 3b — Format into Phi-3 chat template + split train/eval
train_data, eval_data = main.build_hf_dataset(df)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

✅ Dataset  —  train: 360  |  eval: 40

── Sample prompt ──
<|user|>
What is a brokerage account?<|end|>
<|assistant|>
An investment account held at a financial institution that allows you to buy and sell securities.<|end|>



---
## Section 4 — Training v1

| Hyperparameter | Value | Notes |
|---|---|---|
| Epochs | 3 | Increase to 5 if loss still high |
| Learning rate | 2e-4 | Reduce to 1e-4 if loss spikes |
| Batch size | 2 | Safe for T4 and A100 |
| Grad accum | 4 | Effective batch = 8 |

**Healthy loss:** start ~2.5–3.5 → end ~0.3–0.8. Loss < 0.1 = overfitting.

In [9]:
# 4a — Configure trainer
trainer = main.build_trainer(model, tokenizer, train_data, eval_data)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/360 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/40 [00:00<?, ? examples/s]

✅ Trainer ready  —  effective batch: 8  |  epochs: 3  |  lr: 0.0002


In [10]:
# 4b — Launch training (~30 min A100 / ~60 min T4)
metrics = main.run_training(trainer)
print(metrics)

🚀 Training started …  (~30 min A100 / ~60 min T4)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 360 | Num Epochs = 3 | Total steps = 135
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.058460,1.135798
2,0.841125,1.096436
3,0.548558,1.248421


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


── Training complete ──
   final_loss        : 1.0153
   runtime_min       : 4.2
   samples_per_s     : 4.25
   peak_vram_gb      : 2.76
{'final_loss': 1.0153, 'runtime_min': 4.2, 'samples_per_s': 4.25, 'peak_vram_gb': 2.76}


In [11]:
# 4c — Save LoRA adapter v1
main.save_adapter(model, tokenizer, main.ADAPTER_V1)

Unsloth: Restored added_tokens_decoder metadata in phi3-finance-adapter-v1/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi3-finance-adapter-v1.


✅ Adapter saved → phi3-finance-adapter-v1/
   README.md                                   0.0 MB
   adapter_config.json                         0.0 MB
   adapter_model.safetensors                   119.6 MB
   chat_template.jinja                         0.0 MB
   tokenizer.json                              3.6 MB
   tokenizer.model                             0.5 MB
   tokenizer_config.json                       0.0 MB


---
## Section 5 — Inference

In [12]:
# 5a — Single question
answer = main.ask(model, tokenizer, 'What is the 50/30/20 rule?')
print(answer)

Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


A budgeting guideline that allocates 50% of after-tax income to needs, 30% to wants, and 20% to savings and debt repayment.


In [13]:
# 5b — Multi-turn conversation
history = []
reply, history = main.chat_turn(model, tokenizer, 'How do I budget?', history)
print('A:', reply)

reply, history = main.chat_turn(model, tokenizer, 'Give me an example.', history)
print('A:', reply)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Track your income and expenses every month using categories to see where money goes; adjust spending habits accordingly.
A: Every dollar you earn must be allocated to savings, bills, food, entertainment, etc., leaving no room for surprises.


In [14]:
# 5c — Quick sanity test (3 questions)
main.run_sanity_test(model, tokenizer)

Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 Sanity test …

Q: What is the 50/30/20 rule?


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: A budgeting guideline that allocates 50% of after-tax income to needs, 30% to wants, and 20% to savings and debt repayment.
------------------------------------------------------------
Q: How does compound interest work?


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Interest is calculated on both the initial principal and the accumulated interest from previous periods.
------------------------------------------------------------
Q: What is the best way to save money?
A: Automate savings, use high-yield accounts, and cut unnecessary expenses.
------------------------------------------------------------
✅ Sanity test done.


---
## Section 6 — 30-Prompt Evaluation

**Scoring guide:** `2` = correct · `1` = partial · `0` = wrong/refused  
**Target:** pass rate ≥ 75% before merging.

In [15]:
# 6a — Run all 30 prompts (~3-5 min)
results = main.run_evaluation(model, tokenizer)

Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 Running 30-prompt evaluation …

  [01/30] Explain the 50/30/20 budgeting rule in simple terms. …


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [02/30] What is zero-based budgeting and how does it work? …


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [28/30] Why are payday loans considered predatory? …


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [29/30] What does it mean to be underwater on a car loan? …


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [30/30] Is all debt bad? Give examples of good and bad debt. …

✅ All responses collected.


In [16]:
# 6b — Print all responses for manual scoring
main.print_all_responses(results)


Q01: Explain the 50/30/20 budgeting rule in simple terms.
----------------------------------------------------------------------
A guideline where 50% of after-tax income goes to needs, 30% to wants, and 20% is saved
or invested.
Score (0 / 1 / 2): ___

Q02: What is zero-based budgeting and how does it work?
----------------------------------------------------------------------
Allocating every dollar of income to a specific category so that the total spending
equals your income, leaving no money unassigned.
Score (0 / 1 / 2): ___

Q03: What is a sinking fund and when should I use one?
----------------------------------------------------------------------
A dedicated savings account for known, regular expenses like car repairs or insurance;
start it before the cost occurs.
Score (0 / 1 / 2): ___

Q04: What is lifestyle inflation and why is it a problem?
----------------------------------------------------------------------
Gradually increasing spending as income rises, which erodes sa

In [17]:
# 6c — Enter your scores (replace None with 0, 1, or 2)
scores = [
    None, None, None, None, None,   # Q01-Q05
    None, None, None, None, None,   # Q06-Q10
    None, None, None, None, None,   # Q11-Q15
    None, None, None, None, None,   # Q16-Q20
    None, None, None, None, None,   # Q21-Q25
    None, None, None, None, None,   # Q26-Q30
]

summary = main.apply_scores(results, scores)
print(summary)

ℹ️  Only 0/30 filled.
{}


In [18]:
# 6d — See which questions scored 0 (your fix targets)
failures = main.list_failures(results)


❌ 0 question(s) scored 0:


---
## Section 7 — Fix Data & Retrain v2

Add corrected Q&A pairs for every question that scored 0.  
**Golden rule:** fixing 5 bad rows beats adding 50 new ones.

In [19]:
# 7a — Add fixes for failed questions
fixes = [
    (
        'Why are payday loans dangerous?',
        'Payday loans charge APRs of 300-400%+. Borrowers who cannot '
        'repay roll over the loan, paying new fees each time and '
        'trapping themselves in escalating debt.'
    ),
    # ('Your failed question here', 'Better answer here'),
]

df_v2 = main.load_csv()
df_v2 = main.add_fixes(df_v2, fixes)
main.save_csv(df_v2, main.CLEAN_CSV)

📂 Loaded 400 rows  |  Columns: ['Question', 'Answer']
✅ 400 clean rows ready.
✅ 1 fix(es) added. Total: 401 rows.
✅ Saved 'personal_finance_qa_v2.csv' (401 rows).


In [20]:
# 7b — Rebuild dataset from improved CSV
train_data_v2, eval_data_v2 = main.build_hf_dataset(df_v2)

Map:   0%|          | 0/401 [00:00<?, ? examples/s]

✅ Dataset  —  train: 360  |  eval: 41

── Sample prompt ──
<|user|>
What is the all-cash diet?<|end|>
<|assistant|>
A budgeting approach that eliminates credit and debit card use, forcing you to spend only the cash you carry.<|end|>



In [21]:
# 7c — Free old model, reload fresh base + LoRA
main.free_model(model)
model, tokenizer = main.load_base_model()
model = main.apply_lora(model)

✅ GPU memory cleared.
🧠 Loading unsloth/Phi-3-mini-4k-instruct …
==((====))==  Unsloth 2026.7.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded  (dtype=torch.float16)
✅ LoRA applied — 29,884,416 / 2,039,024,640 params trainable (1.47%)


In [22]:
# 7d — Retrain on improved dataset
trainer_v2 = main.build_trainer(
    model, tokenizer, train_data_v2, eval_data_v2,
    output_dir = main.OUTPUT_DIR_V2
)
metrics_v2 = main.run_training(trainer_v2)
print(metrics_v2)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/360 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/41 [00:00<?, ? examples/s]

✅ Trainer ready  —  effective batch: 8  |  epochs: 3  |  lr: 0.0002
🚀 Training started …  (~30 min A100 / ~60 min T4)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 360 | Num Epochs = 3 | Total steps = 135
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Epoch,Training Loss,Validation Loss
1,1.091640,1.141234
2,0.830112,1.073477
3,0.563462,1.185366


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


── Training complete ──
   final_loss        : 1.0217
   runtime_min       : 4.0
   samples_per_s     : 4.47
   peak_vram_gb      : 5.47
{'final_loss': 1.0217, 'runtime_min': 4.0, 'samples_per_s': 4.47, 'peak_vram_gb': 5.47}


In [23]:
# 7e — Save improved adapter v2
main.save_adapter(model, tokenizer, main.ADAPTER_V2)

Unsloth: Restored added_tokens_decoder metadata in phi3-finance-adapter-v2/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi3-finance-adapter-v2.


✅ Adapter saved → phi3-finance-adapter-v2/
   README.md                                   0.0 MB
   adapter_config.json                         0.0 MB
   adapter_model.safetensors                   119.6 MB
   chat_template.jinja                         0.0 MB
   tokenizer.json                              3.6 MB
   tokenizer.model                             0.5 MB
   tokenizer_config.json                       0.0 MB


---
## Section 8 — Merge LoRA into Base Model

> T4 users: if OOM, pass `save_method='merged_4bit'`

In [24]:
# 8a — Merge adapter into base model (~2-3 min)
merged_path = main.merge_adapter(model, tokenizer)
print('Merged model saved to:', merged_path)

🔗 Merging → phi3-finance-merged  (merged_16bit) …


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in phi3-finance-merged/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi3-finance-merged.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...



Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [01:22<01:22, 82.64s/it]

model-00002-of-00002.safetensors: reconstructing file:   0%|          |  0.00B / 2.65GB            

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:50<00:00, 85.28s/it]


Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:31<01:31, 91.27s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:17<00:00, 68.97s/it]


Unsloth: Merge process complete. Saved to `/content/phi3-finance-merged`
✅ Merged model saved  (7.6 GB)
Merged model saved to: phi3-finance-merged


In [25]:
# 8b — [Optional] Push to Hugging Face Hub
# main.push_to_hub(
#     model, tokenizer,
#     hf_username = 'your-username',
#     repo_name   = 'phi3-mini-personal-finance',
#     hf_token    = 'hf_YOUR_TOKEN_HERE',
# )

---
## Section 9 — Gradio Chat UI

`share=True` creates a public HTTPS link valid for 72 hours.

In [26]:
# 9a — Load best model (merged if available, else adapter)
import os
if os.path.exists(main.MERGED_PATH):
    model, tokenizer = main.load_merged_model()
    print('Using merged model')
else:
    model, tokenizer = main.load_model_with_adapter(main.ADAPTER_V2)
    print('Using adapter v2')

==((====))==  Unsloth 2026.7.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Merged model loaded from: phi3-finance-merged
Using merged model


In [27]:
# 9b — Launch Gradio chat UI
main.launch_ui(model, tokenizer, share=True)

🚀 Launching Gradio UI …
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59fe32c51ae194ba06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Section 10 — GCP Vertex AI Deployment (working v11)

Deploys the fine-tuned model to Vertex AI via a custom Docker container. This is the **final working configuration** — every fix from the debugging process is baked in, and the whole section runs top-to-bottom in the correct order.

**The fixes that made it work (all required together):**
1. **Async background loading** — `async def startup()` + `run_in_executor`, so `/health` returns 200 instantly and Vertex AI doesn't kill the container in a restart loop.
2. **Strip the bad tokenizer class** — the saved `tokenizer_config.json` contained `"tokenizer_class": "TokenizersBackend"` which doesn't exist; serve.py removes that key at startup, with a `LlamaTokenizer` fallback.
3. **Remove the duplicate model file in GCS** — the folder had both a sharded model AND a single-file `model.safetensors`, causing a weight-shape collision. Cell 10.1-B deletes the duplicate **before** deploy.
4. **`low_cpu_mem_usage=True`** + **`n1-highmem-8`** (52 GB), GCS SDK download, pinned `transformers==4.46.3` on `python:3.11-slim`, forced stdout logging.

**Run order:** `Auth` → `10.0`(skip if done) → `10.1`(skip if uploaded) → `10.1-B` → `10.2`(skip if built) → `10.3` → `10.4` → `10.5` → `10.6` → `10.7`

### Auth — Run at the start of EVERY Colab session

In [28]:
# AUTH — covers both CLI and Python SDK
# (gcloud auth application-default login CRASHES in Colab — use this)
from google.colab import auth
auth.authenticate_user()

!gcloud config set project project-0853cd1e-1650-41a4-bfd

from google.cloud import storage
client  = storage.Client(project='project-0853cd1e-1650-41a4-bfd')
buckets = [b.name for b in client.list_buckets()]
print('Buckets:', buckets)
print('Auth OK' if buckets else 'Auth FAILED')

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey

Buckets: ['my-unique-phi3-bucket_ss', 'project-0853cd1e-1650-41a4-bfd_cloudbuild']
Auth OK


In [29]:
# Re-upload main.py after any runtime restart
from google.colab import files
files.upload()
import main
print('main.py loaded')

Saving main.py to main (1).py
main.py loaded


### 10.0 — One-Time Setup
> Skip if already done in a previous session.

In [30]:
# 10.0-A — Enable APIs
!gcloud services enable \
  aiplatform.googleapis.com cloudbuild.googleapis.com \
  artifactregistry.googleapis.com storage.googleapis.com \
  --project=project-0853cd1e-1650-41a4-bfd

Operation "operations/acat.p2-16455422825-ff5b42a7-9599-4e96-bc2f-7a799af7351c" finished successfully.


In [31]:
# 10.0-B — Grant ALL required IAM permissions
!gcloud projects add-iam-policy-binding project-0853cd1e-1650-41a4-bfd --member='serviceAccount:16455422825@cloudbuild.gserviceaccount.com' --role='roles/storage.objectAdmin' --quiet
!gcloud projects add-iam-policy-binding project-0853cd1e-1650-41a4-bfd --member='serviceAccount:16455422825@cloudbuild.gserviceaccount.com' --role='roles/cloudbuild.builds.builder' --quiet
!gcloud projects add-iam-policy-binding project-0853cd1e-1650-41a4-bfd --member='serviceAccount:16455422825-compute@developer.gserviceaccount.com' --role='roles/artifactregistry.writer' --quiet
!gcloud projects add-iam-policy-binding project-0853cd1e-1650-41a4-bfd --member='serviceAccount:16455422825-compute@developer.gserviceaccount.com' --role='roles/storage.objectAdmin' --quiet
!gcloud projects add-iam-policy-binding project-0853cd1e-1650-41a4-bfd --member='serviceAccount:16455422825-compute@developer.gserviceaccount.com' --role='roles/logging.logWriter' --quiet
!gcloud projects add-iam-policy-binding project-0853cd1e-1650-41a4-bfd --member='serviceAccount:service-16455422825@gcp-sa-aiplatform.iam.gserviceaccount.com' --role='roles/artifactregistry.reader' --quiet
!gcloud projects add-iam-policy-binding project-0853cd1e-1650-41a4-bfd --member='serviceAccount:16455422825-compute@developer.gserviceaccount.com' --role='roles/artifactregistry.reader' --quiet
print('IAM permissions granted.')

Updated IAM policy for project [project-0853cd1e-1650-41a4-bfd].
bindings:
- members:
  - serviceAccount:service-16455422825@gcp-sa-aiplatform-cc.iam.gserviceaccount.com
  role: roles/aiplatform.customCodeServiceAgent
- members:
  - serviceAccount:service-16455422825@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/aiplatform.serviceAgent
- members:
  - serviceAccount:service-16455422825@gcp-sa-vertex-telemetry.iam.gserviceaccount.com
  role: roles/aiplatform.telemetryServiceAgent
- members:
  - serviceAccount:16455422825-compute@developer.gserviceaccount.com
  - serviceAccount:service-16455422825@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/artifactregistry.reader
- members:
  - serviceAccount:service-16455422825@gcp-sa-artifactregistry.iam.gserviceaccount.com
  role: roles/artifactregistry.serviceAgent
- members:
  - serviceAccount:16455422825-compute@developer.gserviceaccount.com
  role: roles/artifactregistry.writer
- members:
  - serviceAccount:16455422825@cloudb

### 10.1 — Upload Model Weights to GCS
> Skip if already uploaded.

In [32]:
# 10.1 — Upload merged model to GCS
import main
gcs_uri = main.upload_model_to_gcs(
    local_model_dir = main.MERGED_PATH,
    bucket_name     = 'my-unique-phi3-bucket_ss',
    gcs_prefix      = 'models/phi3-finance',
    project_id      = 'project-0853cd1e-1650-41a4-bfd',
)
print('GCS URI:', gcs_uri)

📤 Uploading 'phi3-finance-merged' → gs://my-unique-phi3-bucket_ss/models/phi3-finance …
   ✓ generation_config.json                              0.0 MB
   ✓ config.json                                         0.0 MB
   ✓ tokenizer.model                                     0.5 MB
   ✓ tokenizer_config.json                               0.0 MB
   ✓ chat_template.jinja                                 0.0 MB
   ✓ model-00001-of-00002.safetensors                    4991.4 MB
   ✓ model.safetensors.index.json                        0.0 MB
   ✓ model-00002-of-00002.safetensors                    2650.8 MB
   ✓ tokenizer.json                                      3.6 MB
✅ 9 file(s) uploaded → gs://my-unique-phi3-bucket_ss/models/phi3-finance
GCS URI: gs://my-unique-phi3-bucket_ss/models/phi3-finance


### 10.1-B — Remove Duplicate Model File (critical, run before deploy)

The merged-model folder can contain **both** a sharded model (`model-00001-of-00002` + `model-00002-of-00002`) **and** a single-file `model.safetensors`. Loading both causes a weight-shape collision. Delete the single-file duplicate, keeping the sharded pair + index.

In [33]:
# 10.1-B — List files, then delete the conflicting single-file model.safetensors
from google.cloud import storage
client = storage.Client(project='project-0853cd1e-1650-41a4-bfd')
bucket = client.bucket('my-unique-phi3-bucket_ss')

print('Current model files:')
for blob in client.list_blobs('my-unique-phi3-bucket_ss', prefix='models/phi3-finance'):
    if 'safetensors' in blob.name or 'index' in blob.name:
        print(f'  {blob.name}  ({blob.size/1e6:.1f} MB)')

# Delete the duplicate single-file model (keep model-00001/00002 + index)
dup = bucket.blob('models/phi3-finance/model.safetensors')
if dup.exists():
    dup.delete()
    print('\nDeleted conflicting model.safetensors')
else:
    print('\nNo duplicate found — already clean.')

print('\nRemaining model files:')
for blob in client.list_blobs('my-unique-phi3-bucket_ss', prefix='models/phi3-finance'):
    if 'safetensors' in blob.name or 'index' in blob.name:
        print(f'  {blob.name}  ({blob.size/1e6:.1f} MB)')

Current model files:
  models/phi3-finance/.cache/huggingface/download/model-00001-of-00002.safetensors.metadata  (0.0 MB)
  models/phi3-finance/.cache/huggingface/download/model-00002-of-00002.safetensors.metadata  (0.0 MB)
  models/phi3-finance/.cache/huggingface/download/model.safetensors.index.json.metadata  (0.0 MB)
  models/phi3-finance/model-00001-of-00002.safetensors  (4991.4 MB)
  models/phi3-finance/model-00002-of-00002.safetensors  (2650.8 MB)
  models/phi3-finance/model.safetensors.index.json  (0.0 MB)

No duplicate found — already clean.

Remaining model files:
  models/phi3-finance/.cache/huggingface/download/model-00001-of-00002.safetensors.metadata  (0.0 MB)
  models/phi3-finance/.cache/huggingface/download/model-00002-of-00002.safetensors.metadata  (0.0 MB)
  models/phi3-finance/.cache/huggingface/download/model.safetensors.index.json.metadata  (0.0 MB)
  models/phi3-finance/model-00001-of-00002.safetensors  (4991.4 MB)
  models/phi3-finance/model-00002-of-00002.safete

### 10.2 — Build the v11 Docker Image
> Skip if the v11 image already exists (check with the cell below).

In [34]:
# 10.2-check — does v11 already exist?
!gcloud artifacts docker images list \
  us-central1-docker.pkg.dev/project-0853cd1e-1650-41a4-bfd/phi3-finance-repo \
  --project=project-0853cd1e-1650-41a4-bfd --include-tags \
  --format='table(TAGS,CREATE_TIME)' | grep v11

Listing items under project project-0853cd1e-1650-41a4-bfd, location us-central1, repository phi3-finance-repo.

v11     2026-08-02T09:08:42


In [35]:
# 10.2 — Write serve.py + Dockerfile, build & push v11
import os, ast
SERVING_DIR = './serving'
IMAGE_URI   = 'us-central1-docker.pkg.dev/project-0853cd1e-1650-41a4-bfd/phi3-finance-repo/phi3-finance-server:v11'
os.makedirs(SERVING_DIR, exist_ok=True)

serve_lines = [
    '#!/usr/bin/env python3',
    'import os, sys, json, logging, torch, asyncio',
    'from pathlib import Path',
    'from fastapi import FastAPI, Request, HTTPException',
    'from fastapi.responses import JSONResponse',
    'from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer',
    'from google.cloud import storage as gcs',
    '',
    'logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)',
    'log = logging.getLogger("serve")',
    '',
    'app   = FastAPI()',
    'model = None',
    'tok   = None',
    'ready = False',
    'load_error = None',
    '',
    'GCS_BUCKET = os.environ.get("GCS_BUCKET", "my-unique-phi3-bucket_ss")',
    'GCS_PREFIX = os.environ.get("GCS_PREFIX", "models/phi3-finance")',
    'LOCAL_PATH = "/tmp/model"',
    '',
    '',
    'def download_model():',
    '    log.info("STARTING DOWNLOAD from gs://%s/%s", GCS_BUCKET, GCS_PREFIX)',
    '    sys.stdout.flush()',
    '    client = gcs.Client()',
    '    blobs  = list(client.list_blobs(GCS_BUCKET, prefix=GCS_PREFIX))',
    '    log.info("Found %d blobs", len(blobs))',
    '    sys.stdout.flush()',
    '    if not blobs:',
    '        raise RuntimeError("No files at gs://%s/%s" % (GCS_BUCKET, GCS_PREFIX))',
    '    Path(LOCAL_PATH).mkdir(parents=True, exist_ok=True)',
    '    for blob in blobs:',
    '        rel = blob.name[len(GCS_PREFIX):].lstrip("/")',
    '        if not rel:',
    '            continue',
    '        dest = os.path.join(LOCAL_PATH, rel)',
    '        os.makedirs(os.path.dirname(dest), exist_ok=True)',
    '        log.info("downloading %s (%.1f MB)", blob.name, blob.size/1e6)',
    '        sys.stdout.flush()',
    '        blob.download_to_filename(dest)',
    '    log.info("DOWNLOAD COMPLETE")',
    '    sys.stdout.flush()',
    '',
    '',
    'def fix_tokenizer_config():',
    '    # The saved tokenizer_config.json specifies "tokenizer_class":',
    '    # "TokenizersBackend" which does not exist in transformers. Remove it so',
    '    # AutoTokenizer falls back to the correct Phi-3 (Llama) tokenizer class.',
    '    cfg_path = os.path.join(LOCAL_PATH, "tokenizer_config.json")',
    '    if os.path.exists(cfg_path):',
    '        with open(cfg_path) as f:',
    '            cfg = json.load(f)',
    '        bad = cfg.pop("tokenizer_class", None)',
    '        if bad:',
    '            log.info("Removed bad tokenizer_class=%s from config", bad)',
    '        with open(cfg_path, "w") as f:',
    '            json.dump(cfg, f)',
    '',
    '',
    'def load_model_sync():',
    '    global model, tok, ready, load_error',
    '    try:',
    '        if not os.path.exists(os.path.join(LOCAL_PATH, "config.json")):',
    '            download_model()',
    '        fix_tokenizer_config()',
    '        log.info("LOADING TOKENIZER")',
    '        sys.stdout.flush()',
    '        try:',
    '            tok = AutoTokenizer.from_pretrained(LOCAL_PATH, use_fast=False)',
    '        except Exception as e1:',
    '            log.warning("AutoTokenizer failed (%s), trying LlamaTokenizer", e1)',
    '            tok = LlamaTokenizer.from_pretrained(LOCAL_PATH)',
    '        log.info("LOADING MODEL (low_cpu_mem_usage=True)")',
    '        sys.stdout.flush()',
    '        model = AutoModelForCausalLM.from_pretrained(',
    '            LOCAL_PATH,',
    '            torch_dtype       = torch.float32,',
    '            low_cpu_mem_usage = True,',
    '        )',
    '        model.eval()',
    '        ready = True',
    '        log.info("MODEL READY")',
    '        sys.stdout.flush()',
    '    except Exception as e:',
    '        load_error = str(e)',
    '        log.error("MODEL LOAD FAILED: %s", e, exc_info=True)',
    '        sys.stdout.flush()',
    '',
    '',
    '@app.on_event("startup")',
    'async def startup():',
    '    # CRITICAL: background executor so Uvicorn answers /health immediately.',
    '    # A synchronous startup blocks the server -> Vertex AI kills the container.',
    '    loop = asyncio.get_event_loop()',
    '    loop.run_in_executor(None, load_model_sync)',
    '',
    '',
    '@app.get("/health")',
    'def health():',
    '    return {"status": "healthy", "model_ready": ready, "error": load_error}',
    '',
    '',
    '@app.post("/predict")',
    'async def predict(request: Request):',
    '    if load_error:',
    '        raise HTTPException(status_code=500, detail="Model load failed: " + load_error)',
    '    if not ready:',
    '        raise HTTPException(status_code=503, detail="Model still loading")',
    '    body      = await request.json()',
    '    instances = body.get("instances", [])',
    '    preds     = []',
    '    for inst in instances:',
    '        q      = inst.get("inputs", "")',
    '        params = inst.get("parameters", {})',
    '        prompt = "<|user|>\\n" + q + "<|end|>\\n<|assistant|>\\n"',
    '        enc    = tok(prompt, return_tensors="pt")',
    '        with torch.no_grad():',
    '            out = model.generate(',
    '                **enc,',
    '                max_new_tokens     = int(params.get("max_new_tokens", 150)),',
    '                temperature        = float(params.get("temperature", 0.1)),',
    '                do_sample          = True,',
    '                repetition_penalty = float(params.get("repetition_penalty", 1.1)),',
    '            )',
    '        new_ids = out[0][enc["input_ids"].shape[1]:]',
    '        preds.append({"generated_text": tok.decode(new_ids, skip_special_tokens=True).strip()})',
    '    return JSONResponse({"predictions": preds})',
]
with open(f'{SERVING_DIR}/serve.py', 'w') as f:
    f.write('\n'.join(serve_lines))

docker_lines = [
    'FROM python:3.11-slim',
    'WORKDIR /app',
    'ENV PYTHONUNBUFFERED=1',
    'RUN pip install --no-cache-dir \\',
    '    torch==2.4.1 \\',
    '    transformers==4.46.3 \\',
    '    tokenizers>=0.20.0 \\',
    '    accelerate==0.34.2 \\',
    '    sentencepiece \\',
    '    protobuf \\',
    '    fastapi \\',
    '    uvicorn[standard] \\',
    '    google-cloud-storage',
    'COPY serve.py /app/serve.py',
    'ENV GCS_BUCKET=my-unique-phi3-bucket_ss',
    'ENV GCS_PREFIX=models/phi3-finance',
    'ENV PORT=8080',
    'EXPOSE 8080',
    'CMD ["uvicorn", "serve:app", "--host", "0.0.0.0", "--port", "8080", "--timeout-keep-alive", "600"]',
]
with open(f'{SERVING_DIR}/Dockerfile', 'w') as f:
    f.write('\n'.join(docker_lines))

with open(f'{SERVING_DIR}/serve.py') as f: src = f.read()
ast.parse(src)
print('serve.py syntax OK')
print('Files:', os.listdir(SERVING_DIR))

!gcloud builds submit {SERVING_DIR} --tag={IMAGE_URI} --project=project-0853cd1e-1650-41a4-bfd --machine-type=E2_HIGHCPU_8 --timeout=1200

print('Done. image_uri:', IMAGE_URI)

serve.py syntax OK
Files: ['Dockerfile', 'serve.py']
Creating temporary archive of 2 file(s) totalling 5.1 KiB before compression.
Uploading tarball of [./serving] to [gs://project-0853cd1e-1650-41a4-bfd_cloudbuild/source/1785668758.505468-de46de89b54446a084e36e7b4a2ebfad.tgz]
Created [https://cloudbuild.googleapis.com/v1/projects/project-0853cd1e-1650-41a4-bfd/locations/global/builds/b096bab9-259e-4321-9076-f2d6d0b22889].
Logs are available at [ https://console.cloud.google.com/cloud-build/builds/b096bab9-259e-4321-9076-f2d6d0b22889?project=16455422825 ].
Waiting for build to complete. Polling interval: 1 second(s).
 REMOTE BUILD OUTPUT
starting build "b096bab9-259e-4321-9076-f2d6d0b22889"

FETCHSOURCE
Fetching storage object: gs://project-0853cd1e-1650-41a4-bfd_cloudbuild/source/1785668758.505468-de46de89b54446a084e36e7b4a2ebfad.tgz#1785668758764985
Copying gs://project-0853cd1e-1650-41a4-bfd_cloudbuild/source/1785668758.505468-de46de89b54446a084e36e7b4a2ebfad.tgz#1785668758764985...

### 10.3 — Register v11 Model + Create Endpoint

In [36]:
# 10.3 — Clean up old, register v11, create endpoint
from google.cloud import aiplatform
aiplatform.init(project='project-0853cd1e-1650-41a4-bfd', location='us-central1')

for e in aiplatform.Endpoint.list():
    try:
        e.undeploy_all(); e.delete()
    except Exception as ex:
        print('warn:', ex)
for m in aiplatform.Model.list():
    try:
        m.delete()
    except Exception as ex:
        print('warn:', ex)

IMAGE_URI = 'us-central1-docker.pkg.dev/project-0853cd1e-1650-41a4-bfd/phi3-finance-repo/phi3-finance-server:v11'

vertex_model = aiplatform.Model.upload(
    display_name                = 'phi3-mini-finance-v11',
    serving_container_image_uri = IMAGE_URI,
    serving_container_ports     = [8080],
    serving_container_health_route  = '/health',
    serving_container_predict_route = '/predict',
    serving_container_environment_variables = {
        'GCS_BUCKET': 'my-unique-phi3-bucket_ss',
        'GCS_PREFIX': 'models/phi3-finance',
    },
)
print('Model registered:', vertex_model.resource_name)

endpoint = aiplatform.Endpoint.create(display_name='phi3-finance-endpoint')
print('Endpoint created:', endpoint.resource_name)

Model registered: projects/16455422825/locations/us-central1/models/2167692121962708992
Endpoint created: projects/16455422825/locations/us-central1/endpoints/7541071417601687552


### 10.4 — Deploy v11 (billing starts here)
> CPU on n1-highmem-8 (52 GB RAM). `deploy_request_timeout=1800` gives the container time to download + load the model (~5 min total).

In [39]:
# 10.4 — Deploy v11 on CPU
endpoint.undeploy_all()

deployed = vertex_model.deploy(
    endpoint               = endpoint,
    machine_type           = 'n1-highmem-8',
    min_replica_count      = 1,
    max_replica_count      = 1,
    traffic_percentage     = 100,
    deploy_request_timeout = 1800,
)
print('Deployed v11. Billing is running.')

Deployed v11. Billing is running.


### 10.5 — Watch Startup Logs
Run every 1-2 min after deploy. You'll see: STARTING DOWNLOAD → downloading → DOWNLOAD COMPLETE → Removed bad tokenizer_class → LOADING MODEL → **MODEL READY**.

In [40]:
# 10.5 — Watch container startup logs
import subprocess
from datetime import datetime, timezone, timedelta

cutoff = (datetime.now(timezone.utc) - timedelta(minutes=15)).strftime('%Y-%m-%dT%H:%M:%SZ')
result = subprocess.run([
    'gcloud', 'logging', 'read',
    f'resource.type="aiplatform.googleapis.com/Endpoint" AND timestamp>="{cutoff}" AND NOT jsonPayload.message=~"health"',
    '--project=project-0853cd1e-1650-41a4-bfd',
    '--limit=50',
    '--format=value(timestamp,jsonPayload.message)',
    '--order=asc',
], capture_output=True, text=True)

out = result.stdout
print(out if out.strip() else 'No startup logs yet - wait 1-2 min and run again')

if 'MODEL READY' in out:
    print('\n' + '='*50 + '\nMODEL IS READY - run 10.6 now\n' + '='*50)
elif 'FAILED' in out:
    print('\n' + '='*50 + '\nMODEL LOAD FAILED - traceback is above\n' + '='*50)
elif 'LOADING MODEL' in out:
    print('\nModel loading - wait 2 min')
elif 'DOWNLOAD COMPLETE' in out:
    print('\nDownload done - wait 2 min')
elif 'downloading' in out or 'STARTING' in out:
    print('\nDownloading from GCS - wait 4 min')

2026-08-02T12:06:59.725769996Z	INFO:     Started server process [1]
2026-08-02T12:06:59.725851058Z	INFO:     Waiting for application startup.
2026-08-02T12:06:59.727283Z	INFO:     Application startup complete.
2026-08-02T12:06:59.728054285Z	INFO:serve:STARTING DOWNLOAD from gs://my-unique-phi3-bucket_ss/models/phi3-finance
2026-08-02T12:06:59.729239225Z	INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)
2026-08-02T12:07:00.033914566Z	INFO:serve:Found 14 blobs
2026-08-02T12:07:00.034241676Z	INFO:serve:downloading models/phi3-finance/.cache/huggingface/.gitignore (0.0 MB)
2026-08-02T12:07:00.136323928Z	INFO:serve:downloading models/phi3-finance/.cache/huggingface/CACHEDIR.TAG (0.0 MB)
2026-08-02T12:07:00.200953722Z	INFO:serve:downloading models/phi3-finance/.cache/huggingface/download/model-00001-of-00002.safetensors.metadata (0.0 MB)
2026-08-02T12:07:00.284279346Z	INFO:serve:downloading models/phi3-finance/.cache/huggingface/download/model-00002-of-00002.safetensors

### 10.6 — Test (run once 10.5 shows MODEL IS READY)

In [41]:
# 10.6-A — Ask a question
import main
try:
    answer = main.predict_gcp(endpoint, 'What is the 50/30/20 rule?')
    print('Q: What is the 50/30/20 rule?')
    print('A:', answer)
except Exception as e:
    print('ERROR DETAIL:', str(e))

Q: What is the 50/30/20 rule?
A: A budgeting guideline where 50% of after-tax income goes to needs, 30% to wants, and 20% to savings. <|end|>


In [42]:
# 10.6-B — Run 5 test questions
import main
main.run_gcp_test_predictions(endpoint)

🧪 Testing live endpoint …

Q: What is the 50/30/20 rule?
A: A budgeting guideline where 50% of after-tax income goes to needs, 30% to wants, and 20% to savings. <|end|>
------------------------------------------------------------
Q: How does compound interest work?
A: Interest is calculated on both the initial principal and all accumulated interest from previous periods. <|end|>
------------------------------------------------------------
Q: What is a good credit score?
A: Generally, 670 to 739 is considered good; above 740 is very good. <|end|>
------------------------------------------------------------
Q: Should I pay off debt or invest first?
A: If your interest rates are lower than the market return, focus on building an emergency fund before aggressively saving and investing. Otherwise, prioritize high-interest debt to save more in compound returns over time. <|end|>
------------------------------------------------------------
Q: What is an index fund?
A: A type of investment tha

### 10.7 — Teardown — Always Run When Done
> GCP charges ~$7/day while any endpoint is live. Run this immediately when finished.

In [43]:
# 10.7 — Delete everything - STOPS BILLING
from google.cloud import aiplatform
aiplatform.init(project='project-0853cd1e-1650-41a4-bfd', location='us-central1')
for e in aiplatform.Endpoint.list():
    try:
        e.undeploy_all(); e.delete()
    except Exception as ex:
        print('warn:', ex)
for m in aiplatform.Model.list():
    try:
        m.delete()
    except Exception as ex:
        print('warn:', ex)
print('All resources deleted. Billing stopped.')

All resources deleted. Billing stopped.
